<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026ai_lect05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 人工知能I 第5回：決定木・Random Forest・AdaBoost
## オリベッティ顔データベースによる男女分類

**担当**: 浅川伸一 | **2026年度 前期**

---

## 本日の構成

| セクション | 手法 | キーワード |
|-----------|------|----------|
| 1 | データ準備・不均衡の確認 | 層化分割，ROC-AUC |
| 2 | k-NN | 距離ベース，k の影響 |
| 3 | SVM | マージン最大化，PCA+SVM |
| 4 | 決定木 | ジニ係数，過学習，可視化 |
| 5 | Random Forest | バギング，特徴量重要度ヒートマップ |
| 6 | AdaBoost | ブースティング，誤り修正学習 |
| 7 | 5手法の総合比較 | いつ何を使うか |

**共通データ**: オリベッティ顔データベース（男性0 / 女性1 の二値分類）

**重要**: 女性は40人中3人（30枚/400枚 = 7.5%）→ クラス不均衡に注意

---
## 0. 環境準備

In [ ]:
# Noto Sans CJK JP をインストール
!apt-get install -y fonts-noto-cjk --quiet

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec

# フォントをキャッシュに登録
fm.fontManager.addfont('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')

# 既定フォントに設定
mpl.rc('font', family='Noto Sans CJK JP')

print("現在のフォント:", plt.rcParams['font.family'])


plt.plot((1,2,3))
plt.title('日本語')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve,
    roc_auc_score, ConfusionMatrixDisplay, f1_score
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
# plt.style.use('seaborn-v0_8-darkgrid')
print('環境準備完了')

---
## 1. データ準備と不均衡の確認

### 1.1 オリベッティ顔データベースの読み込みと男女ラベルの付与

In [ ]:
# オリベッティ顔データセットの読み込み
data = fetch_olivetti_faces()
X, y = data.data, data.target

# 男女の判別のため教師データを作成
# 男であれば 0, 女であれば 1 とする
y_sex = np.zeros_like(y)
for woman_start in [70, 310, 340]:
    for i in range(10):
        y_sex[woman_start + i] = 1

print('=== オリベッティ顔データセット（男女二値分類）===')
print(f'  全サンプル数  : {len(X)}')
print(f'  特徴量数     : {X.shape[1]}（64×64 ピクセル）')
print(f'  男性（0）    : {(y_sex==0).sum()} 枚（{(y_sex==0).mean()*100:.1f}%）')
print(f'  女性（1）    : {(y_sex==1).sum()} 枚（{(y_sex==1).mean()*100:.1f}%）')
print()
print('⚠️  ナイーブ分類器（全員男性と予測）のaccuracy:', f'{(y_sex==0).mean()*100:.1f}%')
print('   → accuracy だけでは評価できない！')

### 1.2 サンプル画像の確認

In [ ]:
# 男性・女性のサンプル画像を表示
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
male_idx   = np.where(y_sex == 0)[0]
female_idx = np.where(y_sex == 1)[0]

for i, idx in enumerate(male_idx[:8]):
    axes[0, i].imshow(X[idx].reshape(64, 64), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('男性', fontsize=11, pad=4)

for i, idx in enumerate(female_idx[:8]):
    axes[1, i].imshow(X[idx].reshape(64, 64), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('女性', fontsize=11, pad=4)

plt.suptitle('オリベッティ顔データセット：男性（上）・女性（下）', fontsize=13)
plt.tight_layout()
plt.show()

### 1.3 PCA前処理と層化分割

**PCAを使う理由**
- 4096次元 → 100次元（計算速度と汎化性能を改善）
- `whiten=True`：各主成分の分散を1に正規化（SVMとの相性が良い）
- **全手法で共通のPCA変換を使用**（公平な比較のため）

**層化分割を使う理由**
- 女性30枚を訓練・テストに適切に配分するため（`stratify=y_sex`）

In [ ]:
# 層化分割（stratify=y_sex で女性サンプルを均等配分）
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y_sex, test_size=0.25, stratify=y_sex, random_state=42
)

# PCA前処理（whiten=True：各主成分の分散を1に正規化）
N_COMPONENTS = 100
pca = PCA(n_components=N_COMPONENTS, whiten=True, random_state=42)
X_train = pca.fit_transform(X_train_raw)  # 訓練データでfitしてtransform
X_test  = pca.transform(X_test_raw)       # テストデータはtransformのみ

print('=== データ分割 ===')
print(f'  訓練データ: {len(X_train)} 枚')
print(f'    男性: {(y_train==0).sum()} 枚, 女性: {(y_train==1).sum()} 枚')
print(f'  テストデータ: {len(X_test)} 枚')
print(f'    男性: {(y_test==0).sum()} 枚, 女性: {(y_test==1).sum()} 枚')
print(f'\nPCA後の次元数: {X_train.shape[1]}')

# 固有顔（主成分）の可視化
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(pca.components_[i].reshape(64, 64), cmap='RdBu_r')
    ax.set_title(f'固有顔 {i+1}\n寄与率={pca.explained_variance_ratio_[i]*100:.1f}%',
                 fontsize=9)
    ax.axis('off')
plt.suptitle('固有顔（上位10主成分）：PCAが抽出した顔の基底パターン', fontsize=12)
plt.tight_layout()
plt.show()

cum_var = np.cumsum(pca.explained_variance_ratio_)
print(f'\n上位{N_COMPONENTS}主成分が説明する分散: {cum_var[-1]*100:.1f}%')

### 1.4 評価関数の定義（全セクション共通）

accuracyだけでなく，女性クラスのPrecision・Recall・F1・ROC-AUCを確認します。

In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba=None):
    """
    男女分類の評価指標を表示する共通関数
    - accuracy は参考値（不均衡データでは過信禁物）
    - 女性クラス（1）の Recall が重要
    """
    print(f'\n{'='*50}')
    print(f'  {name}')
    print('='*50)
    print(classification_report(y_true, y_pred,
                                target_names=['男性(0)','女性(1)']))
    if y_proba is not None:
        auc = roc_auc_score(y_true, y_proba)
        print(f'ROC-AUC: {auc:.4f}')
    return {
        'accuracy': (y_true==y_pred).mean(),
        'f1_female': f1_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_proba) if y_proba is not None else None
    }

def plot_confusion_roc(y_true, y_pred, y_proba, title, ax1, ax2):
    """混同行列とROC曲線を横並びで描画"""
    ConfusionMatrixDisplay(
        confusion_matrix(y_true, y_pred),
        display_labels=['男性','女性']
    ).plot(ax=ax1, colorbar=False)
    ax1.set_title(f'{title}\n混同行列', fontsize=11)

    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        auc = roc_auc_score(y_true, y_proba)
        ax2.plot(fpr, tpr, linewidth=2, label=f'AUC={auc:.3f}')
        ax2.plot([0,1],[0,1],'gray',linestyle='--',linewidth=1)
        ax2.set_xlabel('偽陽性率(FPR)'); ax2.set_ylabel('真陽性率(TPR)')
        ax2.set_title(f'{title}\nROC曲線', fontsize=11)
        ax2.legend()

# 結果を収集する辞書
results = {}
print('評価関数の定義完了')

---
## 2. k-NN（k近傍法）

**直観**：テストサンプルに最も近いk個の訓練サンプルの多数決でクラスを決定

**注意点**：多数派クラス（男性370枚）に引きずられる → k とクラス不均衡の関係を確認

In [ ]:
# k の値を変えて Female の Recall を確認
k_vals = [1, 3, 5, 7, 10, 15, 20]
knn_recall_female = []
knn_auc = []

for k in k_vals:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred_k = knn.predict(X_test)
    y_prob_k = knn.predict_proba(X_test)[:, 1]
    recall_f = (y_pred_k[y_test==1]==1).mean()  # female recall
    auc_k    = roc_auc_score(y_test, y_prob_k)
    knn_recall_female.append(recall_f)
    knn_auc.append(auc_k)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(k_vals, knn_recall_female, 'tomato', marker='o', linewidth=2)
axes[0].set_xlabel('k（近傍数）', fontsize=11)
axes[0].set_ylabel('女性の Recall', fontsize=11)
axes[0].set_title('k と女性 Recall の関係\nk が大きいほど多数派（男性）に引きずられる', fontsize=11)
axes[0].set_xticks(k_vals)

axes[1].plot(k_vals, knn_auc, 'steelblue', marker='o', linewidth=2)
axes[1].set_xlabel('k（近傍数）', fontsize=11)
axes[1].set_ylabel('ROC-AUC', fontsize=11)
axes[1].set_title('k と ROC-AUC の関係', fontsize=11)
axes[1].set_xticks(k_vals)

plt.tight_layout()
plt.show()

best_k = k_vals[np.argmax(knn_auc)]
print(f'最良 k（AUC基準）: k = {best_k}')

In [ ]:
# 最良 k で評価
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
y_pred_knn  = knn_best.predict(X_test)
y_proba_knn = knn_best.predict_proba(X_test)[:, 1]

results['k-NN'] = evaluate_model('k-NN', y_test, y_pred_knn, y_proba_knn)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_confusion_roc(y_test, y_pred_knn, y_proba_knn, f'k-NN (k={best_k})',
                  axes[0], axes[1])
plt.tight_layout()
plt.show()

### 2.1 2次元 PCA 空間での決定境界
4096次元を2次元に圧縮して，決定境界を可視化します。

In [ ]:
# 2次元 PCA で決定境界を可視化
pca2 = PCA(n_components=2, whiten=True, random_state=42)
X_tr2 = pca2.fit_transform(X_train_raw)
X_te2 = pca2.transform(X_test_raw)

# メッシュグリッド
h = 0.5
x_min, x_max = X_tr2[:,0].min()-2, X_tr2[:,0].max()+2
y_min, y_max = X_tr2[:,1].min()-2, X_tr2[:,1].max()+2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

knn2 = KNeighborsClassifier(n_neighbors=best_k)
knn2.fit(X_tr2, y_train)
Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
colors = {0: 'steelblue', 1: 'tomato'}
for cls, label in zip([0,1],['男性','女性']):
    mask_tr = y_train == cls
    plt.scatter(X_tr2[mask_tr,0], X_tr2[mask_tr,1],
                c=colors[cls], s=40, alpha=0.6, label=f'訓練:{label}')
    mask_te = y_test == cls
    plt.scatter(X_te2[mask_te,0], X_te2[mask_te,1],
                c=colors[cls], s=100, marker='*', edgecolors='k',
                linewidth=0.8, label=f'テスト:{label}')
plt.xlabel('第1主成分（固有顔1）', fontsize=11)
plt.ylabel('第2主成分（固有顔2）', fontsize=11)
plt.title(f'k-NN (k={best_k}) の決定境界（PCA2次元空間）', fontsize=12)
plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()

---
## 3. SVM（サポートベクターマシン）

**直観**：クラス間のマージン（余裕）を最大化する決定境界を求める

高次元・少数サンプル問題に強く，顔認識との相性が最も良い手法

※ sklearn 公式の顔認識チュートリアルと同じ構成（PCA + SVM）

In [ ]:
from sklearn.model_selection import GridSearchCV

# ハイパーパラメータ探索（C と γ）
param_grid = {
    'C':     [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 'scale'],
}
svm_cv = GridSearchCV(
    SVC(kernel='rbf', class_weight='balanced', probability=True),
    param_grid, cv=StratifiedKFold(5), scoring='roc_auc'
)
svm_cv.fit(X_train, y_train)

print(f'最適パラメータ: {svm_cv.best_params_}')
print(f'CV AUC:        {svm_cv.best_score_:.4f}')

y_pred_svm  = svm_cv.predict(X_test)
y_proba_svm = svm_cv.predict_proba(X_test)[:, 1]

results['SVM'] = evaluate_model('SVM (RBF)', y_test, y_pred_svm, y_proba_svm)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_confusion_roc(y_test, y_pred_svm, y_proba_svm, 'SVM (RBF)', axes[0], axes[1])
plt.tight_layout()
plt.show()

### 3.1 サポートベクターの確認

In [ ]:
best_svm = svm_cv.best_estimator_
n_sv = best_svm.n_support_
print(f'サポートベクター数: 男性 {n_sv[0]} 枚, 女性 {n_sv[1]} 枚')
print(f'全訓練サンプルに占める割合: {sum(n_sv)/len(X_train)*100:.1f}%')
print()
print('解釈：')
print('  決定境界の近くにある（判断が難しい）サンプルのみが決定に寄与')
print('  残りのサンプルを削除してもモデルは変わらない')

---
## 4. 決定木（Decision Tree）

**直観**：「if-then ルール」の木構造で分類

**本セクションのねらい**：決定木が高次元・少数データに弱いことを確認し，RF への動機づけとする

In [ ]:
# 深さを変えて訓練・テスト精度（女性 F1）を比較
depths = [1, 2, 3, 4, 5, 7, 10, None]
dt_train_f1, dt_test_f1, dt_test_auc = [], [], []

for d in depths:
    dt = DecisionTreeClassifier(
        max_depth=d, class_weight='balanced', random_state=42
    )
    dt.fit(X_train, y_train)
    # 訓練
    y_tr_pred = dt.predict(X_train)
    dt_train_f1.append(f1_score(y_train, y_tr_pred))
    # テスト
    y_te_pred = dt.predict(X_test)
    y_te_prob = dt.predict_proba(X_test)[:, 1]
    dt_test_f1.append(f1_score(y_test, y_te_pred))
    dt_test_auc.append(roc_auc_score(y_test, y_te_prob))

depth_labels = [str(d) if d else 'None\n(制限なし)' for d in depths]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(depths))
axes[0].plot(x, dt_train_f1, 'steelblue', marker='o', linewidth=2, label='訓練 F1（女性）')
axes[0].plot(x, dt_test_f1,  'tomato',    marker='o', linewidth=2, label='テスト F1（女性）')
axes[0].set_xticks(x); axes[0].set_xticklabels(depth_labels)
axes[0].set_xlabel('max_depth', fontsize=11)
axes[0].set_ylabel('F1スコア（女性クラス）', fontsize=11)
axes[0].set_title('決定木の深さと過学習\n深くなると訓練↑ テスト↓（過学習）', fontsize=11)
axes[0].legend()

axes[1].plot(x, dt_test_auc, 'green', marker='s', linewidth=2)
axes[1].set_xticks(x); axes[1].set_xticklabels(depth_labels)
axes[1].set_xlabel('max_depth', fontsize=11)
axes[1].set_ylabel('ROC-AUC（テスト）', fontsize=11)
axes[1].set_title('決定木の深さと ROC-AUC', fontsize=11)

plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(dt_test_auc)]
print(f'最良 max_depth（AUC基準）: {best_depth}')

In [ ]:
# 最良深さの決定木を可視化
dt_best = DecisionTreeClassifier(
    max_depth=3, class_weight='balanced', random_state=42
)
dt_best.fit(X_train, y_train)
y_pred_dt  = dt_best.predict(X_test)
y_proba_dt = dt_best.predict_proba(X_test)[:, 1]

results['決定木'] = evaluate_model('決定木 (max_depth=3)', y_test, y_pred_dt, y_proba_dt)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_confusion_roc(y_test, y_pred_dt, y_proba_dt, '決定木 (depth=3)', axes[0], axes[1])
plt.tight_layout()
plt.show()

# 決定木の構造を可視化
fig, ax = plt.subplots(figsize=(18, 6))
pca_feat_names = [f'PC{i+1}' for i in range(N_COMPONENTS)]
plot_tree(dt_best, feature_names=pca_feat_names,
          class_names=['男性','女性'],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('決定木の構造（max_depth=3）\n各ノードは「主成分Xの値 > 閾値か？」で分岐', fontsize=12)
plt.tight_layout()
plt.show()

print('\n観察ポイント：')
print('  - 各ノードの分割基準は「PCX > 閾値」（固有顔成分の強さ）')
print('  - 女性30枚は少数なため，深い木では過学習しやすい')
print('  - Random Forest はこの問題を複数の木で解決する')

---
## 5. Random Forest

**直観**：多数の決定木を並列学習し，多数決で分類（バギング）

**ポイント**：各木で使う特徴量もランダムに選ぶ → 木同士の多様性が鍵

**本セクションのハイライト**：`feature_importances_` を顔のヒートマップとして可視化

In [ ]:
# Random Forest の学習
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

results['RF'] = evaluate_model('Random Forest', y_test, y_pred_rf, y_proba_rf)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_confusion_roc(y_test, y_pred_rf, y_proba_rf, 'Random Forest', axes[0], axes[1])
plt.tight_layout()
plt.show()

### 5.1 特徴量重要度ヒートマップ

`feature_importances_` はPCA主成分ごとの重要度。
これを元の画像空間に逆変換すると**「どのピクセルが男女識別に重要か」**が可視化できます。

In [ ]:
# PCA 主成分の重要度 → 元の画像空間に逆変換
importances = rf.feature_importances_  # shape: (n_components,)

# 重要度を重みとして固有顔を線形結合 → 画素空間の重要度マップ
importance_face = np.dot(importances, pca.components_)  # (4096,)
importance_map  = importance_face.reshape(64, 64)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 左：主成分ごとの重要度（棒グラフ，上位20）
top20 = np.argsort(importances)[-20:][::-1]
axes[0].bar(range(20), importances[top20], color='steelblue', alpha=0.8)
axes[0].set_xticks(range(20))
axes[0].set_xticklabels([f'PC{i+1}' for i in top20], rotation=45, fontsize=8)
axes[0].set_xlabel('主成分', fontsize=11)
axes[0].set_ylabel('重要度', fontsize=11)
axes[0].set_title('主成分ごとの重要度（上位20）', fontsize=11)

# 中：重要度マップ（画素空間に逆変換）
im = axes[1].imshow(np.abs(importance_map), cmap='hot', interpolation='nearest')
plt.colorbar(im, ax=axes[1])
axes[1].set_title('男女識別に重要なピクセル\n（明るいほど重要）', fontsize=11)
axes[1].axis('off')

# 右：平均顔との重ね合わせ
mean_face = X_train_raw.mean(axis=0).reshape(64, 64)
axes[2].imshow(mean_face, cmap='gray', alpha=0.8)
axes[2].imshow(np.abs(importance_map), cmap='hot', alpha=0.4, interpolation='nearest')
axes[2].set_title('平均顔 + 重要度オーバーレイ\n（識別的な顔領域の可視化）', fontsize=11)
axes[2].axis('off')

plt.suptitle('Random Forest：どのピクセルが男女識別に重要か', fontsize=13)
plt.tight_layout()
plt.show()

print('観察ポイント：')
print('  - 輪郭，眉，目などの領域が重要になっていることが多い')
print('  - 回帰の係数（線形な重要度）と違い，非線形な寄与も捉えている')

In [ ]:
# 木の数（n_estimators）と性能の関係
n_estimators_list = [1, 5, 10, 30, 50, 100, 200, 300]
rf_auc_list = []

for n in n_estimators_list:
    rf_n = RandomForestClassifier(
        n_estimators=n, class_weight='balanced',
        max_features='sqrt', random_state=42, n_jobs=-1
    )
    rf_n.fit(X_train, y_train)
    prob = rf_n.predict_proba(X_test)[:, 1]
    rf_auc_list.append(roc_auc_score(y_test, prob))

plt.figure(figsize=(8, 4))
plt.semilogx(n_estimators_list, rf_auc_list, 'steelblue', marker='o', linewidth=2)
plt.xlabel('木の数（n_estimators）', fontsize=11)
plt.ylabel('ROC-AUC（テスト）', fontsize=11)
plt.title('RF：木の数と ROC-AUC\n木が増えると安定するが，増やしすぎても効果は頭打ち', fontsize=11)
plt.tight_layout()
plt.show()

---
## 6. AdaBoost

**直観**：前の弱学習器が**誤分類したサンプルに重みを増やして**次の弱学習器を学習

「苦手なものを集中的に練習する」＝第4回で学んだ誤り修正学習のアナロジー

**バギング（RF）との対比**
- RF：木を並列に独立学習 → 多数決
- AdaBoost：木を順番に学習 → 誤りに注目しながら修正

In [ ]:
# 不均衡対策：女性サンプルに大きな初期重みを設定（class_weight に相当）
n_female = (y_train == 1).sum()
n_male   = (y_train == 0).sum()
sample_weight = np.where(y_train == 1,
                         n_male / n_female,  # 女性の重みを男性比率分増加
                         1.0)
sample_weight /= sample_weight.sum()  # 正規化

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # 弱学習器：深さ1の決定木
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)
ada.fit(X_train, y_train, sample_weight=sample_weight)

y_pred_ada  = ada.predict(X_test)
y_proba_ada = ada.predict_proba(X_test)[:, 1]

results['AdaBoost'] = evaluate_model('AdaBoost', y_test, y_pred_ada, y_proba_ada)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_confusion_roc(y_test, y_pred_ada, y_proba_ada, 'AdaBoost', axes[0], axes[1])
plt.tight_layout()
plt.show()

### 6.1 学習曲線：弱学習器の累積とAUCの変化

In [ ]:
# 弱学習器（ステージ）数ごとの AUC を記録
staged_auc = []
for proba_stage in ada.staged_predict_proba(X_test):
    staged_auc.append(roc_auc_score(y_test, proba_stage[:, 1]))

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(staged_auc)+1), staged_auc, 'tomato', linewidth=1.5)
plt.xlabel('弱学習器の数（ステージ）', fontsize=11)
plt.ylabel('ROC-AUC（テスト）', fontsize=11)
plt.title('AdaBoost 学習曲線\n弱学習器を加えるごとに性能が向上 → 過学習に注意', fontsize=11)
plt.axhline(results['RF']['auc'], color='steelblue',
            linestyle='--', linewidth=1.5, label=f"RF AUC={results['RF']['auc']:.3f}")
plt.legend()
plt.tight_layout()
plt.show()

print(f'最終 AUC: {staged_auc[-1]:.4f}')
print(f'最大 AUC: {max(staged_auc):.4f}（{np.argmax(staged_auc)+1} ステージ）')

### 6.2 サンプル重みの変化（ブースティングの直観）

In [ ]:
# 最初の数ステージでのサンプル重みを追跡（AdaBoostの内部動作を可視化）
# AdaBoostClassifier の estimator_weights_ で弱学習器の重みを確認
weak_weights = ada.estimator_weights_
weak_errors  = ada.estimator_errors_

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左：弱学習器ごとの重み（α）
axes[0].plot(range(1, len(weak_weights)+1), weak_weights,
             'steelblue', linewidth=1.2, alpha=0.7)
axes[0].set_xlabel('弱学習器のステージ番号', fontsize=11)
axes[0].set_ylabel('弱学習器の重み（α）', fontsize=11)
axes[0].set_title('各弱学習器の重み\n精度が高い弱学習器ほど大きな重みで最終決定に参加', fontsize=11)

# 右：弱学習器ごとのエラー率
axes[1].plot(range(1, len(weak_errors)+1), weak_errors,
             'tomato', linewidth=1.2, alpha=0.7)
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=1,
                label='ランダム（0.5）')
axes[1].set_xlabel('弱学習器のステージ番号', fontsize=11)
axes[1].set_ylabel('弱学習器のエラー率', fontsize=11)
axes[1].set_title('各弱学習器のエラー率\n0.5未満 = ランダムよりは良い（弱学習器の条件）', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

print('AdaBoostのポイント：')
print('  各弱学習器はランダムより「少しだけ」良ければ十分')
print('  それらを積み重ねることで強い分類器を作る')
print('  ← 心理学的学習（誤り修正）のアナロジー')

---
## 7. 5手法の総合比較：いつ何を使うか

### 7.1 評価指標の比較

In [ ]:
# 結果テーブルの作成
compare_df = pd.DataFrame({
    '手法':      list(results.keys()),
    'Accuracy':  [r['accuracy'] for r in results.values()],
    'F1（女性）': [r['f1_female'] for r in results.values()],
    'ROC-AUC':   [r['auc'] for r in results.values()],
})
print('=== 5手法の比較（オリベッティ男女分類）===')
print(compare_df.round(4).to_string(index=False))

# 棒グラフで比較
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics   = ['Accuracy', 'F1（女性）', 'ROC-AUC']
colors    = ['#e41a1c','#377eb8','#4daf4a','#ff7f00','#984ea3']
methods   = compare_df['手法'].tolist()

for ax, metric in zip(axes, metrics):
    vals = compare_df[metric].values
    bars = ax.bar(methods, vals, color=colors, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, 1.15)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('5手法の評価指標比較（オリベッティ男女分類）', fontsize=13)
plt.tight_layout()
plt.show()

### 7.2 ROC曲線の重ね合わせ

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
roc_data = [
    ('k-NN',    y_proba_knn, '#e41a1c'),
    ('SVM',     y_proba_svm, '#377eb8'),
    ('決定木',   y_proba_dt,  '#4daf4a'),
    ('RF',      y_proba_rf,  '#ff7f00'),
    ('AdaBoost',y_proba_ada, '#984ea3'),
]
for name, proba, color in roc_data:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'gray',linestyle='--',linewidth=1,label='ランダム')
ax.set_xlabel('偽陽性率（FPR）', fontsize=12)
ax.set_ylabel('真陽性率（TPR）', fontsize=12)
ax.set_title('5手法のROC曲線比較\n（女性を見逃さないために TPR を高く保つことが重要）', fontsize=12)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 7.3 「いつ何を使うか」の判断軸

In [ ]:
print('='*60)
print('  5手法の特徴まとめ（オリベッティ男女分類の結果から）')
print('='*60)
guide = """
k-NN
  ✅ 実装が簡単，直観的
  ⚠️  予測時に全訓練データと比較 → データが増えると遅い
  ⚠️  高次元データ（生ピクセル）では距離が意味を失いやすい
  → PCAと組み合わせると改善

SVM
  ✅ 高次元・少数サンプルに強い（顔認識の定番）
  ✅ class_weight='balanced' で不均衡に対処できる
  ⚠️  ハイパーパラメータ（C, γ）の調整が必要
  → PCA + SVM が顔認識の標準パイプライン

決定木
  ✅ 可視化・解釈がしやすい
  ✅ スケール不変（標準化不要）
  ⚠️  少数サンプル・高次元で過学習しやすい
  ⚠️  顔のような複雑な分類には不向き
  → RFへの橋渡しとして理解する

Random Forest
  ✅ 過学習しにくい（決定木の弱点を克服）
  ✅ feature_importances_ → 重要ピクセルの可視化
  ✅ 並列学習が可能（速い）
  → 構造化データのデフォルト選択肢

AdaBoost
  ✅ 弱学習器の組み合わせで強い分類器
  ✅ 誤り修正学習のアナロジー（心理学的に面白い）
  ⚠️  ノイズや外れ値に敏感
  ⚠️  RFより過学習しやすい
  → 比較的クリーンなデータで有効
"""
print(guide)

---
## 8. まとめと準備学習課題

In [ ]:
print('='*60)
print('  第5回まとめ：5手法の整理')
print('='*60)
print("""
【距離ベース】
  k-NN：最近傍の多数決。k が大きいほど滑らかな境界

【マージンベース】
  SVM：マージン最大化 + RBFカーネルで非線形境界
       顔認識（高次元・少数サンプル）に最も相性が良い

【木ベース】
  決定木：if-then ルール。解釈しやすいが過学習しやすい
  RF    ：バギング（並列・多数決）で過学習を抑制
  AdaBoost：ブースティング（逐次・誤り修正）で弱学習器を強化

【クラス不均衡への対処（本データのポイント）】
  → accuracy だけでは評価できない
  → F1（女性）・ROC-AUC を主指標に使う
  → class_weight='balanced' / sample_weight で対処

【次回：教師なし学習と次元削減】
  PCA, SVD, NMF, SOM, k-means
  → 本日の PCA 前処理がなぜ有効だったかを理論的に理解する
""")

---
## 準備学習課題

**課題1**（必須・60分）  
教師なし学習が心理学研究で役立つ場面を1つ挙げ，その理由を説明せよ（100字程度）。

**課題2**（必須・60分）  
情報量（エントロピー）と交差エントロピーを，具体的な数値例を用いて説明せよ。
（ヒント：第4回で学んだ交差エントロピー損失とどう関係するか）

---
### 解答欄

**課題1**:


**課題2**:


---
## 参考文献

1. Eriksen, C. W. (2011). Sklearn Olivetti Faces Recognition Example. https://scikit-learn.org/stable/auto_examples/applications/plot_face_recognition.html
2. Freund, Y., & Schapire, R. E. (1997). A decision-theoretic generalization of on-line learning. *Journal of Computer and System Sciences, 55*(1), 119–139.
3. Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32.
4. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.

---
*人工知能I 第5回実習ノートブック | 担当：浅川伸一*